# PaySense — Engine 2: BNPL Risk Classification
**Goal:** Train Random Forest, XGBoost, and LightGBM on synthetic BNPL scenarios. Select the model with the highest F1 + ROC-AUC (Recall prioritised) on the held-out test set.

**Sequential architecture:** Engine 1 forecasted cash flow is used as an input feature here, linking the two engines.

| Step | Description |
|------|-------------|
| 0 | Setup: install packages, mount Drive |
| 1 | Imports & configuration |
| 2 | Generate synthetic BNPL dataset |
| 3 | Exploratory Data Analysis |
| 4 | Feature engineering |
| 5 | Train / val / test split (70/15/15) + SMOTE |
| 6 | Random Forest |
| 7 | XGBoost |
| 8 | LightGBM |
| 9 | Model comparison & winner selection |
| 10 | SHAP explainability on winner |
| 11 | Save winner model |

## 0. Setup

In [ ]:
!pip install -q xgboost lightgbm imbalanced-learn shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT = '/content/drive/MyDrive/PaySense/'
MODEL_DIR  = DRIVE_ROOT + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)
print("Drive mounted. Model dir:", MODEL_DIR)

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, joblib, json
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             f1_score, precision_score, recall_score, accuracy_score,
                             roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

import random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Libraries loaded.")
print(f"  XGBoost  : {xgb.__version__}")
print(f"  LightGBM : {lgb.__version__}")

## 2. Generate Synthetic BNPL Dataset
2,000 synthetic Gen-Z Malaysian student/early-career profiles.
The `forecasted_cash_flow` feature simulates Engine 1 output, demonstrating the sequential pipeline.

In [ ]:
N = 2000
rng = np.random.default_rng(SEED)

# ── Demographic & income features ─────────────────────────────────────────────
age              = rng.integers(18, 31, N)
employment       = rng.choice([0, 1, 2], N, p=[0.45, 0.35, 0.20])
                 # 0=student, 1=part-time, 2=full-time

base_income = np.where(employment == 0,
                  rng.uniform(300, 800, N),
              np.where(employment == 1,
                  rng.uniform(700, 1800, N),
                  rng.uniform(1500, 4000, N)))
monthly_income = np.round(base_income, 2)

# ── Spending behaviour ─────────────────────────────────────────────────────────
expense_ratio     = rng.uniform(0.6, 1.3, N)
monthly_expenses  = np.round(monthly_income * expense_ratio, 2)

# ── BNPL usage ────────────────────────────────────────────────────────────────
num_bnpl_plans    = rng.integers(0, 7, N)
bnpl_outstanding  = np.round(num_bnpl_plans * rng.uniform(50, 400, N), 2)
missed_payments   = rng.integers(0, 7, N)
# Higher BNPL stress → more missed payments (realistic correlation)
missed_payments   = np.clip(missed_payments + (bnpl_outstanding / monthly_income * 1.5).astype(int), 0, 10).astype(int)

# ── Engine 1 linkage: simulated 3-month average forecasted net cash flow ──────
noise             = rng.normal(0, 150, N)
forecasted_cf     = np.round((monthly_income - monthly_expenses) * rng.uniform(0.7, 1.1, N) + noise, 2)

# ── Derived ratios ────────────────────────────────────────────────────────────
income_expense_ratio = np.round(monthly_income / np.maximum(monthly_expenses, 1), 4)
bnpl_income_ratio    = np.round(bnpl_outstanding / np.maximum(monthly_income, 1), 4)
savings_rate         = np.round((monthly_income - monthly_expenses) / np.maximum(monthly_income, 1), 4)

# ── Target: BNPL Risk (0=low, 1=high) ────────────────────────────────────────
# Probabilistic risk based on multiple stress signals
risk_score = (
    (missed_payments >= 2).astype(int) * 3 +
    (bnpl_income_ratio > 0.5).astype(int) * 2 +
    (savings_rate < 0).astype(int) * 2 +
    (income_expense_ratio < 0.9).astype(int) * 1 +
    (forecasted_cf < -100).astype(int) * 2 +
    (num_bnpl_plans >= 4).astype(int) * 1
)
risk_prob    = 1 / (1 + np.exp(-(risk_score - 3.5)))
bnpl_risk    = (rng.uniform(0, 1, N) < risk_prob).astype(int)

df = pd.DataFrame({
    'age':                  age,
    'employment_status':    employment,
    'monthly_income':       monthly_income,
    'monthly_expenses':     monthly_expenses,
    'num_bnpl_plans':       num_bnpl_plans,
    'bnpl_outstanding':     bnpl_outstanding,
    'missed_payments':      missed_payments,
    'forecasted_cash_flow': forecasted_cf,
    'income_expense_ratio': income_expense_ratio,
    'bnpl_income_ratio':    bnpl_income_ratio,
    'savings_rate':         savings_rate,
    'bnpl_risk':            bnpl_risk
})

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nClass distribution:")
print(df['bnpl_risk'].value_counts().rename({0:'Low Risk (0)', 1:'High Risk (1)'}))
print(f"\nClass balance: {df['bnpl_risk'].mean()*100:.1f}% high-risk")
df.head()

In [ ]:
df.to_csv(MODEL_DIR + 'bnpl_synthetic_dataset.csv', index=False)
print("Saved: bnpl_synthetic_dataset.csv")
df.describe().round(2)

## 3. Exploratory Data Analysis

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig)

# Class balance pie
ax0 = fig.add_subplot(gs[0, 0])
counts = df['bnpl_risk'].value_counts()
ax0.pie(counts.values, labels=['Low Risk','High Risk'], colors=['#2ecc71','#e74c3c'],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize':10})
ax0.set_title('Class Distribution')

# Income by risk
ax1 = fig.add_subplot(gs[0, 1])
df.boxplot(column='monthly_income', by='bnpl_risk', ax=ax1,
           boxprops=dict(color='#2c3e50'), medianprops=dict(color='red'))
ax1.set_title('Monthly Income by Risk Class')
ax1.set_xlabel('BNPL Risk (0=Low, 1=High)')
plt.sca(ax1); plt.title('Monthly Income by Risk')

# Missed payments distribution
ax2 = fig.add_subplot(gs[0, 2])
for risk, color, label in [(0,'#2ecc71','Low Risk'), (1,'#e74c3c','High Risk')]:
    subset = df[df['bnpl_risk'] == risk]['missed_payments']
    ax2.hist(subset, bins=range(0, 12), alpha=0.6, color=color, label=label, density=True)
ax2.set_title('Missed Payments Distribution')
ax2.set_xlabel('Missed Payments (last 6 months)')
ax2.legend()

# BNPL-to-income ratio
ax3 = fig.add_subplot(gs[1, 0])
df.boxplot(column='bnpl_income_ratio', by='bnpl_risk', ax=ax3)
ax3.set_title('BNPL-to-Income Ratio by Risk')
ax3.set_xlabel('BNPL Risk (0=Low, 1=High)')
plt.sca(ax3); plt.title('BNPL/Income Ratio by Risk')

# Forecasted cash flow (Engine 1 linkage)
ax4 = fig.add_subplot(gs[1, 1])
for risk, color, label in [(0,'#2ecc71','Low Risk'), (1,'#e74c3c','High Risk')]:
    subset = df[df['bnpl_risk'] == risk]['forecasted_cash_flow']
    ax4.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
ax4.axvline(0, color='black', linestyle='--', alpha=0.5)
ax4.set_title('Forecasted Cash Flow (Engine 1 Output)')
ax4.set_xlabel('Forecasted Net Cash Flow (RM)')
ax4.legend()

# Savings rate
ax5 = fig.add_subplot(gs[1, 2])
for risk, color, label in [(0,'#2ecc71','Low Risk'), (1,'#e74c3c','High Risk')]:
    subset = df[df['bnpl_risk'] == risk]['savings_rate']
    ax5.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
ax5.axvline(0, color='black', linestyle='--', alpha=0.5)
ax5.set_title('Savings Rate by Risk Class')
ax5.set_xlabel('Savings Rate')
ax5.legend()

# Correlation heatmap
ax6 = fig.add_subplot(gs[2, :])
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax6, cmap='RdYlGn', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5, annot_kws={'size':7})
ax6.set_title('Feature Correlation Matrix')

plt.suptitle('PaySense — Engine 2 EDA', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(MODEL_DIR + 'engine2_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering

In [ ]:
FEATURES = [
    'age', 'employment_status',
    'monthly_income', 'monthly_expenses',
    'num_bnpl_plans', 'bnpl_outstanding', 'missed_payments',
    'forecasted_cash_flow',
    'income_expense_ratio', 'bnpl_income_ratio', 'savings_rate'
]
TARGET = 'bnpl_risk'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"Features : {len(FEATURES)}")
print(f"Samples  : {len(X)}")
print(f"\nFeature list:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")

## 5. Train / Val / Test Split + SMOTE
- Split: 70% train, 15% validation, 15% test (stratified)
- SMOTE applied on **train set only** — val and test remain untouched (real-world distribution)

In [ ]:
# 70 / 15 / 15 stratified split
X_temp, X_test,  y_temp, y_test  = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val,  y_train, y_val  = train_test_split(X_temp, y_temp, test_size=0.15/0.85,
                                                     random_state=SEED, stratify=y_temp)
print(f"Train : {len(X_train):4d} rows  |  High-risk: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Val   : {len(X_val):4d} rows  |  High-risk: {y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"Test  : {len(X_test):4d} rows  |  High-risk: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

In [ ]:
# SMOTE on train only
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE — Train: {len(X_train_sm)} rows")
print(f"  Low Risk  (0): {(y_train_sm==0).sum()}")
print(f"  High Risk (1): {(y_train_sm==1).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(axes,
    [y_train.value_counts(), y_train_sm.value_counts()],
    ['Before SMOTE', 'After SMOTE']):
    ax.bar(['Low Risk','High Risk'], [counts.get(0,0), counts.get(1,0)],
           color=['#2ecc71','#e74c3c'], alpha=0.8)
    ax.set_title(title); ax.set_ylabel('Count')
plt.suptitle('SMOTE Effect on Training Set', fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'smote_effect.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def evaluate_classifier(model, X_val, y_val, X_test, y_test, model_name):
    y_val_pred   = model.predict(X_val)
    y_test_pred  = model.predict(X_test)
    y_test_prob  = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_test_pred)
    prec = precision_score(y_test, y_test_pred, zero_division=0)
    rec  = recall_score(y_test, y_test_pred, zero_division=0)
    f1   = f1_score(y_test, y_test_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_test_prob)

    print(f"\n{'='*48}  {model_name}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}   ← prioritised")
    print(f"  F1        : {f1:.4f}   ← primary selection metric")
    print(f"  ROC-AUC   : {auc:.4f}   ← primary selection metric")
    print(f"\n{classification_report(y_test, y_test_pred, target_names=['Low Risk','High Risk'])}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Low','High'], yticklabels=['Low','High'])
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

    fpr, tpr, _ = roc_curve(y_test, y_test_prob)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{model_name} — ROC Curve')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(MODEL_DIR + f'{model_name.lower().replace(" ","_")}_eval.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    return {
        'model': model_name,
        'Accuracy':  round(acc, 4),
        'Precision': round(prec, 4),
        'Recall':    round(rec, 4),
        'F1':        round(f1, 4),
        'ROC_AUC':   round(auc, 4),
        'F1_AUC_avg': round((f1 + auc) / 2, 4)
    }

## 6. Model 1 — Random Forest

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
model_rf.fit(X_train_sm, y_train_sm)
print("Random Forest fitted.")
print(f"OOB not enabled (use cross-val for generalisation estimate)")

# 5-fold CV on train set
cv_scores = cross_val_score(model_rf, X_train_sm, y_train_sm, cv=5, scoring='f1', n_jobs=-1)
print(f"5-Fold CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
rf_metrics = evaluate_classifier(model_rf, X_val, y_val, X_test, y_test, 'Random Forest')

## 7. Model 2 — XGBoost

In [ ]:
scale_pos = (y_train_sm == 0).sum() / (y_train_sm == 1).sum()

model_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1, verbosity=0
)
model_xgb.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print("XGBoost fitted.")

cv_scores_xgb = cross_val_score(model_xgb, X_train_sm, y_train_sm, cv=5, scoring='f1', n_jobs=-1)
print(f"5-Fold CV F1: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")

In [ ]:
xgb_metrics = evaluate_classifier(model_xgb, X_val, y_val, X_test, y_test, 'XGBoost')

## 8. Model 3 — LightGBM

In [ ]:
model_lgb = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1
)
model_lgb.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
print("LightGBM fitted.")

cv_scores_lgb = cross_val_score(model_lgb, X_train_sm, y_train_sm, cv=5, scoring='f1', n_jobs=-1)
print(f"5-Fold CV F1: {cv_scores_lgb.mean():.4f} ± {cv_scores_lgb.std():.4f}")

In [ ]:
lgb_metrics = evaluate_classifier(model_lgb, X_val, y_val, X_test, y_test, 'LightGBM')

## 9. Model Comparison & Winner Selection
**Selection criterion:** Highest average of F1 + ROC-AUC, with Recall as a tiebreaker.

In [ ]:
results_df = pd.DataFrame([rf_metrics, xgb_metrics, lgb_metrics])
results_df = results_df.sort_values('F1_AUC_avg', ascending=False).reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df)+1))
results_df['Winner'] = ['✓ SELECTED' if i==0 else '' for i in range(len(results_df))]

print("\n" + "="*70)
print("   ENGINE 2 — MODEL EVALUATION RESULTS")
print("="*70)
print(results_df[['Rank','model','Accuracy','Precision','Recall','F1','ROC_AUC','F1_AUC_avg','Winner']]
      .to_string(index=False))
print("="*70)

WINNER_NAME  = results_df.iloc[0]['model']
WINNER_F1    = results_df.iloc[0]['F1']
WINNER_AUC   = results_df.iloc[0]['ROC_AUC']
WINNER_REC   = results_df.iloc[0]['Recall']
print(f"\nWinner  : {WINNER_NAME}")
print(f"F1      : {WINNER_F1}   |   ROC-AUC : {WINNER_AUC}   |   Recall : {WINNER_REC}")

In [ ]:
# Side-by-side metric comparison
metric_cols = ['Accuracy','Precision','Recall','F1','ROC_AUC']
x = np.arange(len(metric_cols))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (_, row) in enumerate(results_df[['model'] + metric_cols].iterrows()):
    vals = [row[m] for m in metric_cols]
    bars = ax.bar(x + i*width, vals, width, label=row['model'], color=colors[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title(f'Engine 2 — Model Comparison  |  Winner: {WINNER_NAME}', fontweight='bold')
ax.legend()
ax.axhline(0.8, color='grey', linestyle=':', alpha=0.4, label='0.8 baseline')
plt.tight_layout()
plt.savefig(MODEL_DIR+'engine2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

results_df.to_csv(MODEL_DIR+'engine2_results.csv', index=False)
print("Comparison chart and results table saved.")

## 10. SHAP Explainability on Winner
SHAP (SHapley Additive exPlanations) shows which features push a prediction toward high risk or low risk.

In [ ]:
winner_model = {'Random Forest': model_rf, 'XGBoost': model_xgb, 'LightGBM': model_lgb}[WINNER_NAME]

# Tree explainer works for all three candidates
explainer   = shap.TreeExplainer(winner_model)
shap_values = explainer.shap_values(X_test)

# For binary classifiers some libraries return list [class0, class1]
if isinstance(shap_values, list):
    sv = shap_values[1]   # class 1 = High Risk
else:
    sv = shap_values

print(f"SHAP values computed for {WINNER_NAME}")
print(f"Shape: {sv.shape}  (samples x features)")

In [ ]:
# ── Global: Feature Importance (mean |SHAP|) ─────────────────────────────────
shap_importance = pd.DataFrame({
    'feature': FEATURES,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ['#e74c3c' if v > shap_importance['mean_abs_shap'].median()
              else '#3498db' for v in shap_importance['mean_abs_shap']]
ax.barh(shap_importance['feature'], shap_importance['mean_abs_shap'],
        color=colors_bar, alpha=0.85)
ax.set_title(f'SHAP Feature Importance — {WINNER_NAME}\n(Mean |SHAP Value| across test set)',
             fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig(MODEL_DIR+'shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(shap_importance.sort_values('mean_abs_shap', ascending=False).to_string(index=False))

In [ ]:
# ── SHAP Summary (beeswarm) ──────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=FEATURES, show=False, plot_size=None)
plt.title(f'SHAP Summary Plot — {WINNER_NAME}', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(MODEL_DIR+'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Waterfall for a high-risk prediction ────────────────────────────────
y_test_prob = winner_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.where((winner_model.predict(X_test) == 1) & (y_test_prob > 0.75))[0]

if len(high_risk_idx) > 0:
    sample_idx = high_risk_idx[0]
    print(f"Explaining sample #{sample_idx} — predicted HIGH RISK (prob={y_test_prob[sample_idx]:.3f})")
    print("\nFeature values:")
    for f, v in zip(FEATURES, X_test.iloc[sample_idx].values):
        print(f"  {f:<25}: {v}")

    exp = shap.Explanation(
        values=sv[sample_idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
        data=X_test.iloc[sample_idx].values,
        feature_names=FEATURES
    )
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(exp, show=False)
    plt.title(f'SHAP Waterfall — High-Risk Profile (prob={y_test_prob[sample_idx]:.3f})',
              fontweight='bold')
    plt.tight_layout()
    plt.savefig(MODEL_DIR+'shap_waterfall_highrisk.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No high-confidence high-risk samples found in test set.")

## 11. Save Winner Model

In [ ]:
joblib.dump(winner_model, MODEL_DIR+'engine2_winner.pkl')
print(f"Saved: engine2_winner.pkl  ({WINNER_NAME})")

np.save(MODEL_DIR+'engine2_feature_names.npy', np.array(FEATURES))
print("Saved: engine2_feature_names.npy")

meta = {
    'engine': 2, 'task': 'bnpl_risk_classification', 'winner': WINNER_NAME,
    'selection_criterion': 'highest_avg_F1_and_ROCAUC_recall_prioritised',
    'features': FEATURES,
    'results': {
        'Random Forest': rf_metrics,
        'XGBoost':       xgb_metrics,
        'LightGBM':      lgb_metrics
    },
    'data': {
        'source':   'synthetic_bnpl_scenarios (N=2000)',
        'split':    '70/15/15 train/val/test, stratified',
        'smote':    'applied on train set only'
    },
    'generated_at': datetime.now().isoformat()
}
with open(MODEL_DIR+'engine2_metadata.json','w') as f:
    json.dump(meta, f, indent=2)

results_df.to_csv(MODEL_DIR+'engine2_results.csv', index=False)

print("\n" + "="*55)
print(f"  ENGINE 2 COMPLETE — Winner: {WINNER_NAME}")
print(f"  F1      : {WINNER_F1}")
print(f"  ROC-AUC : {WINNER_AUC}")
print(f"  Recall  : {WINNER_REC}")
print("="*55)
print("\nBoth engines complete. Models saved to:", MODEL_DIR)
print("Next: integrate Engine 1 + Engine 2 into PaySense app pipeline.")